# Stage 2 Saturation Analysis

Plots the owned v1 saturation algorithm: drive-normalised tanh plus a drive-coupled dynamic low-pass model. This mirrors `Source/DSP/Saturation.cpp` for curve and spectrum sanity checks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fs = 48_000
max_clip_drive = 5.0
min_clip_drive = 0.0001
lpf_max_hz = 20_000
lpf_min_hz = 6_000

def shape(x, drive):
    if drive <= min_clip_drive:
        return x
    clip_drive = min_clip_drive + drive * (max_clip_drive - min_clip_drive)
    return np.tanh(clip_drive * x) / np.tanh(clip_drive)

def cutoff_hz(drive):
    return lpf_max_hz * ((lpf_min_hz / lpf_max_hz) ** drive)

x = np.linspace(-1.25, 1.25, 4096)
for drive in [0.0, 0.25, 0.5, 0.75, 1.0]:
    plt.plot(x, shape(x, drive), label=f'drive={drive:.2f}')
plt.title('Drive-normalised tanh transfer')
plt.xlabel('input')
plt.ylabel('output')
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()

drive_values = np.linspace(0, 1, 256)
plt.plot(drive_values, cutoff_hz(drive_values))
plt.title('Dynamic LPF cutoff mapping')
plt.xlabel('drive amount')
plt.ylabel('cutoff Hz')
plt.grid(True, alpha=0.25)
plt.show()

In [ ]:
duration = 1.0
t = np.arange(int(fs * duration)) / fs
sine = 0.8 * np.sin(2 * np.pi * 1_000 * t)

plt.figure(figsize=(10, 4))
for drive in [0.25, 0.5, 1.0]:
    y = shape(sine, drive)
    spectrum = np.fft.rfft(y * np.hanning(len(y)))
    freqs = np.fft.rfftfreq(len(y), 1 / fs)
    db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12))
    plt.plot(freqs, db - db.max(), label=f'drive={drive:.2f}')
plt.xlim(0, 12_000)
plt.ylim(-120, 5)
plt.title('1 kHz sine harmonic spectrum')
plt.xlabel('frequency Hz')
plt.ylabel('relative dB')
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()